In [1]:
import subprocess

print("===== nvidia-smi =====")

try:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    print(result.stdout)

except Exception as e:
    print("nvidia-smi 실패:", e)

print("\n===== PyTorch CUDA =====")

import torch

print("torch          :", torch.__version__)
print("CUDA build     :", torch.version.cuda)
print("is_available   :", torch.cuda.is_available())
print("device count   :", torch.cuda.device_count())

assert torch.cuda.is_available(), "❌ CUDA를 사용할 수 없습니다."

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)

    print(
        f"  [{i}] {p.name} | "
        f"{p.total_memory / 1024**3:.1f} GB | "
        f"cc {p.major}.{p.minor}"
    )

print("\n===== 실연산 테스트 =====")

x = torch.randn(4096, 4096, device="cuda")
y = x @ x

torch.cuda.synchronize()

print(
    "✅ GPU 연산 성공:",
    y.shape,
    "| 메모리:",
    f"{torch.cuda.memory_allocated()/1024**2:.0f} MB",
)

del x
del y

torch.cuda.empty_cache()

print("✅ 테스트 완료")

===== nvidia-smi =====
0, NVIDIA GeForce RTX 5060 Laptop GPU, 474 MiB, 8151 MiB, 0 %


===== PyTorch CUDA =====
torch          : 2.10.0+cu128
CUDA build     : 12.8
is_available   : True
device count   : 1
  [0] NVIDIA GeForce RTX 5060 Laptop GPU | 8.0 GB | cc 12.0

===== 실연산 테스트 =====
✅ GPU 연산 성공: torch.Size([4096, 4096]) | 메모리: 136 MB
✅ 테스트 완료
